# Memory + Skills demo

End-to-end walkthrough of the memory + skills surface using a real OpenAI LLM. Streamed via `agent.run_stream(...)` and rendered with the `EventConsole` (`stream_events`) — you'll see system prompts, tool calls, streamed text deltas, and the `<system-reminder>` attachments live.

Requires `OPENAI_API_KEY` in the environment. Each step builds its own `LLMAgent` to keep cells independently runnable — state lives on disk in a session-scoped tmpdir.

**What this covers**

1. Agent reads a hand-curated `MEMORY.md` injected into the system prompt.
2. Agent authors a new topic memory **with the generic file tools** (`Read` / `Write` / `Edit`) and updates `MEMORY.md` to link the new topic.
3. Agent loads and follows a Skill on demand via `load_skill`.
4. Per-turn memory relevance via `memory_relevance_attachment` + a selector — watch the `<system-reminder>` block land on the user message.

**Important architectural change**: there are no specialized memory tools (no `save_memory` / `load_memory` / etc.). Memory files are just markdown files in a directory; the agent reads and writes them with the same `Read` / `Write` / `Edit` it would use on any other file. This mirrors Claude Code's design, where the system prompt teaches the format and the file tools do the I/O.

## Imports

In [ ]:
from collections.abc import Sequence
from pathlib import Path
from typing import Any
import tempfile

from grasp_agents import (
    FileMemoryProvider,
    LLMAgent,
    MemoryEntry,
    ProcPacketOutEvent,
    RunContext,
    SkillRegistry,
    load_skill,
    stream_events,
)
from grasp_agents.tools.file_edit import FileEditToolkit
from grasp_agents.llm_providers.openai_completions.completions_llm import (
    OpenAILLM,
    OpenAILLMSettings,
)


def make_llm() -> OpenAILLM:
    return OpenAILLM(
        model_name="openai/gpt-4o-mini",
        llm_settings=OpenAILLMSettings(temperature=0.0),
    )


async def run_and_capture(agent: LLMAgent, message: str, ctx: RunContext):
    """Stream an agent with the EventConsole and return the final payload."""
    final = None
    async for event in stream_events(
        agent.run_stream(message, ctx=ctx),
        show_input_messages=True,
        max_input_msg_lines=150,
        max_tool_output_lines=150,
    ):
        if isinstance(event, ProcPacketOutEvent):
            final = event.data.payloads[0]
    return final

## Setup

Create a session-scoped tmpdir holding a tiny memdir + a skills tree. Same layout as `~/.grasp/projects/<sanitized-cwd>/memory/` on disk — `MEMORY.md` plus topic `.md` files with YAML frontmatter.

The `MEMORY.md` index references topic memories as markdown links: `[name](file.md) — description`. The link text is the topic `name`; the href is the file path that the file tools use. The agent maintains this index when it authors a new topic in Step 2.

In [ ]:
TMPDIR = Path(tempfile.mkdtemp(prefix="memory_skills_demo_"))
MEMDIR = TMPDIR / "memdir"
SKILLS_ROOT = TMPDIR / "skills"
MEMDIR.mkdir()

(MEMDIR / "MEMORY.md").write_text(
    "# grasp-agents demo memory index\n\n"
    "Topics:\n\n"
    "- [user_preferences](user_preferences.md) \u2014 how the user likes their answers formatted.\n",
    encoding="utf-8",
)
# The body of `user_preferences` lives ONLY in user_preferences.md.
# MEMORY.md just links to it; the agent reads it with the Read tool.
(MEMDIR / "user_preferences.md").write_text(
    "---\nname: user_preferences\ntype: user\n"
    "description: how the user likes their answers formatted\n---\n"
    "Reply in concise bullet points unless the user asks for prose.\n",
    encoding="utf-8",
)

skill_dir = SKILLS_ROOT / "summarize"
skill_dir.mkdir(parents=True)
(skill_dir / "SKILL.md").write_text(
    "---\nname: summarize\n"
    "description: Summarize text in exactly 3 bullets.\n---\n"
    "Output exactly three bullet points capturing the key facts of "
    "the text the user supplied in the current turn. No preamble, "
    "no closing remark, no tool calls \u2014 just the three bullets.\n",
    encoding="utf-8",
)

print("memdir:", MEMDIR)
print("files:", sorted(p.name for p in MEMDIR.iterdir()))

## Step 1 \u2014 agent reads `MEMORY.md` and a linked topic

`FileMemoryProvider(MEMDIR)` populates `ctx.memory`, and `enable_memory=True` turns on the memory feature on the agent. That triggers:

- The `memory` system-prompt section (injects `MEMORY.md` verbatim under a `<memory-index>` block + the substrate instructions).
- The `memory_relevance_attachment` (per-turn surfacing, used in Step 4).

Memory is **opt-in** at the agent level \u2014 the default is `enable_memory=False` so users don't silently get memory machinery added to their system prompts. Same story for `enable_skills` in Step 3.

We wire just the `Read` tool here: the index in the system prompt tells the agent *what* topic files exist; it follows the link to read the body it needs. Step 2 wires the full `Write` / `Edit` toolkit for authoring.

In [ ]:
provider = FileMemoryProvider(MEMDIR)
toolkit = provider.make_file_toolkit()

agent = LLMAgent[str, str, None](
    name="demo",
    llm=make_llm(),
    enable_memory=True,
    # Read-only step: wire Read alone so the agent can fetch the
    # topic body that MEMORY.md points to. Step 2 wires Write+Edit
    # for the authoring loop.
    tools=[toolkit.read],
    sys_prompt="You are a helpful assistant.",
    env_info=False,
    stream_llm=True,
)
ctx: RunContext[None] = RunContext(state=None, memory=provider)

final = await run_and_capture(
    agent,
    "How do I prefer my answers formatted? Look it up if you need to.",
    ctx,
)
print("\nfinal:", final)

## Step 2 — author a topic + maintain the `MEMORY.md` index

Memory authoring uses the **generic file tools** rooted at the memdir. The agent calls `Write` to create the topic `.md` file (with YAML frontmatter the system prompt taught it) and `Edit` to add a link line to `MEMORY.md`. No specialized memory tools are involved.

`provider.make_file_toolkit()` constructs a `FileEditToolkit` rooted at the provider's memdir — saves wiring the path twice. For an MCP-backed provider (companion notebook) the same call returns a toolkit backed by an `MCPFileBackend`; the agent code is identical.

In [ ]:
provider = FileMemoryProvider(MEMDIR)
toolkit = provider.make_file_toolkit()

agent = LLMAgent[str, str, None](
    name="demo",
    llm=make_llm(),
    enable_memory=True,
    tools=toolkit.tools(),  # Read / Write / Edit
    sys_prompt="You are a helpful assistant.",
    env_info=False,
    stream_llm=True,
)
ctx = RunContext(state=None, memory=provider)

final = await run_and_capture(
    agent,
    "Please remember: I'm based in Berlin (CET timezone) and "
    "I work on ML research full-time.",
    ctx,
)
print("\nfinal:", final)
print("\nmemdir now:", sorted(p.name for p in MEMDIR.iterdir()))
print("\nMEMORY.md after the run:")
print((MEMDIR / "MEMORY.md").read_text())

## Step 3 — agent uses a Skill via `load_skill`

The auto-attached `skills` section renders the catalog (`<available_skills>`) into the system prompt. The agent sees the `summarize` skill listed and calls `load_skill("summarize")` to fetch its body, then follows the body's instructions.

In [ ]:
agent = LLMAgent[str, str, None](
    name="demo",
    llm=make_llm(),
    # enable_skills=True turns on the skills section + auto-attaches
    # the `load_skill` tool. `list_skills` stays opt-in (catalog is
    # already in the system prompt).
    enable_skills=True,
    tools=[],
    sys_prompt=(
        "You are a helpful assistant. When the user mentions a "
        "skill by name, call `load_skill` exactly once to retrieve "
        "its body, then follow the body's instructions in your "
        "next reply."
    ),
    max_turns=4,
    env_info=False,
    stream_llm=True,
)
print("wired tools:", list(agent.tools))
ctx = RunContext(
    state=None,
    memory=FileMemoryProvider(MEMDIR),
    skills=SkillRegistry.from_path(SKILLS_ROOT),
)

final = await run_and_capture(
    agent,
    "Use the summarize skill on this text:\n"
    "Memory in grasp-agents lives in a markdown directory at "
    "~/.grasp/projects/<sanitized-cwd>/memory/. MEMORY.md is "
    "always loaded into the system prompt. Topic .md files have "
    "YAML frontmatter and are loaded on demand with the generic "
    "Read tool.",
    ctx,
)
print("\nfinal:", final)

## Step 4 — per-turn memory relevance via a selector

Register a `Selector[MemoryEntry]` on the provider. The auto-registered `memory_relevance_attachment` runs at user-message-build time, fetches the bodies of the entries the selector picked, and appends them to the user message wrapped in `<system-reminder>`. The system prompt above stays cache-stable; only the user-side block changes turn to turn.

Set `show_input_messages=True` on the console (which our helper does) to see the attached block appear next to the user message.

In this demo the selector trivially keeps all `type=user` entries. In a real app you'd plug in an LLM-judged sideQuery, embeddings, or a domain rule here.

In [ ]:
provider = FileMemoryProvider(MEMDIR)


def keep_user_type(
    *, entries: Sequence[MemoryEntry], **_: Any
) -> Sequence[MemoryEntry]:
    return [e for e in entries if e.memory_type == "user"]


provider.set_selector(keep_user_type)

agent = LLMAgent[str, str, None](
    name="demo",
    llm=make_llm(),
    enable_memory=True,
    sys_prompt="You are a helpful assistant.",
    env_info=False,
    stream_llm=True,
)
ctx = RunContext(state=None, memory=provider)

final = await run_and_capture(
    agent,
    "Tell me one thing about how I prefer my answers.",
    ctx,
)
print("\nfinal:", final)

## Cleanup

Optional — remove the tmpdir.

In [ ]:
import shutil

shutil.rmtree(TMPDIR, ignore_errors=True)
print("removed:", TMPDIR)